**Task 3.1 — Features, train/test split and baseline**

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the cleaned appointment data
df = pd.read_csv("appointments_clean.csv")

# Display the first five rows
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [2]:
# Select only the allowed features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert reminder_sent from Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Convert categorical features into one-hot encoded columns
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Display feature columns and shape
print("Feature columns:")
print(X.columns.tolist())

print("\nFeature shape:", X.shape)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Feature shape: (990, 11)


In [3]:
# Target variable
y = df["no_show"]

print("Target distribution:")
print(y.value_counts())

print("\nNo-show percentage:")
print(round(y.mean() * 100, 2), "%")

Target distribution:
no_show
0    772
1    218
Name: count, dtype: int64

No-show percentage:
22.02 %


In [4]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display dataset sizes
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Count no-shows in the test set
print("Test set no-shows:", int(y_test.sum()))

Training rows: 792
Testing rows: 198
Test set no-shows: 49


In [5]:
# Baseline: always predict Attended (0)
baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))

Baseline accuracy: 0.7525


**Task 3.2 — Logistic Regression**

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [7]:
# Create the scaler
scaler = StandardScaler()

# Learn scaling parameters from training data only
X_train_s = scaler.fit_transform(X_train)

# Apply the same parameters to test data
X_test_s = scaler.transform(X_test)

In [8]:
# Create the model
model = LogisticRegression(max_iter=1000)

# Train using scaled training data
model.fit(X_train_s, y_train)

# Predict test-set classes
y_pred = model.predict(X_test_s)

# Predict test-set probabilities
y_prob = model.predict_proba(X_test_s)[:, 1]

In [9]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Precision
precision = precision_score(y_test, y_pred)

# Recall
recall = recall_score(y_test, y_pred)

# F1 score
f1 = f1_score(y_test, y_pred)

# ROC-AUC
auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 3))
print("\nConfusion Matrix:\n", cm)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1 Score:", round(f1, 3))
print("ROC-AUC:", round(auc, 3))

Accuracy: 0.788

Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789


In [10]:
# Display each feature alongside its coefficient
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

print(
    coefficients.sort_values(
        "coefficient",
        ascending=True
    )
)

                    feature  coefficient
5             reminder_sent    -0.580669
0                       age    -0.253880
8   booking_channel_Walk-In    -0.186364
6       booking_channel_App    -0.046804
10     visit_type_Follow-up    -0.015658
9   visit_type_Consultation     0.015658
7     booking_channel_Phone     0.215881
1               distance_km     0.253448
2                   fee_inr     0.452761
3                 lead_days     0.614471
4            prior_no_shows     0.649612


In [11]:
# Find the reminder_sent feature position
reminder_index = list(X.columns).index("reminder_sent")

# Get its scaled coefficient
reminder_coef = model.coef_[0][reminder_index]

# Odds ratio per one standard deviation
scaled_odds_ratio = np.exp(reminder_coef)

print("Scaled coefficient:", reminder_coef)
print("Odds ratio per SD:", scaled_odds_ratio)

Scaled coefficient: -0.5806686033659922
Odds ratio per SD: 0.559524141750829


In [12]:
# Get the standard deviation used by the scaler
reminder_sd = scaler.scale_[reminder_index]

# Convert scaled coefficient to original 0-to-1 feature change
original_change_coef = reminder_coef / reminder_sd

# Calculate odds ratio for No (0) to Yes (1)
reminder_odds_ratio = np.exp(original_change_coef)

print("Reminder standard deviation:", reminder_sd)
print("Original change coefficient:", original_change_coef)
print("No-to-Yes odds ratio:", reminder_odds_ratio)

Reminder standard deviation: 0.4949752621616814
Original change coefficient: -1.1731265130909097
No-to-Yes odds ratio: 0.3093980903124433


**Task 3.3 — Decision Tree and Overfitting**

In [13]:
from sklearn.tree import DecisionTreeClassifier

In [14]:
# Create a decision tree without a depth limit
dt_full = DecisionTreeClassifier(random_state=42)

# Train the model
dt_full.fit(X_train, y_train)

# Predictions on training data
dt_train_pred = dt_full.predict(X_train)

# Predictions on test data
dt_test_pred = dt_full.predict(X_test)

# Calculate accuracy
dt_full_train_acc = accuracy_score(y_train, dt_train_pred)
dt_full_test_acc = accuracy_score(y_test, dt_test_pred)

print("Full-depth tree")
print("Training accuracy:", round(dt_full_train_acc, 3))
print("Test accuracy:", round(dt_full_test_acc, 3))
print("Tree depth:", dt_full.get_depth())

Full-depth tree
Training accuracy: 1.0
Test accuracy: 0.672
Tree depth: 15


In [15]:
# Create a decision tree with maximum depth of 3
dt_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
dt_depth3.fit(X_train, y_train)

# Predictions
dt3_train_pred = dt_depth3.predict(X_train)
dt3_test_pred = dt_depth3.predict(X_test)

# Calculate accuracy
dt3_train_acc = accuracy_score(y_train, dt3_train_pred)
dt3_test_acc = accuracy_score(y_test, dt3_test_pred)

print("Depth-3 tree")
print("Training accuracy:", round(dt3_train_acc, 3))
print("Test accuracy:", round(dt3_test_acc, 3))
print("Tree depth:", dt_depth3.get_depth())

Depth-3 tree
Training accuracy: 0.822
Test accuracy: 0.778
Tree depth: 3


In [16]:
tree_comparison = pd.DataFrame({
    "Model": ["Full-depth tree", "Depth-3 tree"],
    "Training accuracy": [dt_full_train_acc, dt3_train_acc],
    "Test accuracy": [dt_full_test_acc, dt3_test_acc]
})

print(tree_comparison)

             Model  Training accuracy  Test accuracy
0  Full-depth tree            1.00000       0.671717
1     Depth-3 tree            0.82197       0.777778


The full-depth tree achieves perfect training accuracy but much lower test accuracy, indicating overfitting. Limiting the depth to 3 reduces memorisation and improves generalisation to unseen data.

**Task 3.4 — Random Forest and Model Comparison**

In [17]:
from sklearn.ensemble import RandomForestClassifier

In [18]:
# Create a Random Forest with 100 trees
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train using original, unscaled features
rf.fit(X_train, y_train)

# Predictions
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest training accuracy:",
      round(accuracy_score(y_train, rf.predict(X_train)), 3))

print("Random Forest test accuracy:",
      round(accuracy_score(y_test, rf_pred), 3))

print("Random Forest recall:",
      round(recall_score(y_test, rf_pred), 3))

print("Random Forest ROC-AUC:",
      round(roc_auc_score(y_test, rf_prob), 3))

Random Forest training accuracy: 1.0
Random Forest test accuracy: 0.747
Random Forest recall: 0.306
Random Forest ROC-AUC: 0.706


In [19]:
importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(importances.head(11))

                    feature  importance
3                 lead_days    0.219905
0                       age    0.196788
1               distance_km    0.194739
2                   fee_inr    0.137400
4            prior_no_shows    0.105709
5             reminder_sent    0.045711
7     booking_channel_Phone    0.026913
8   booking_channel_Walk-In    0.019735
9   visit_type_Consultation    0.019364
6       booking_channel_App    0.018394
10     visit_type_Follow-up    0.015341


In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, dt3_test_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, dt3_test_pred),
        recall_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, dt_depth3.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(comparison.round(3))

                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


Based on the test-set reporting comparison, Logistic Regression appears to perform best, with the highest ROC-AUC and good accuracy. It also offers better interpretability than the Random Forest. However, this is only a descriptive comparison. The final model will be selected using training-only cross-validation in next Task, without using test-set results for model selection."

# Task 3.5 — Tune, choose the model and the threshold, then test once

In [21]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

In [22]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

# IMPORTANT: Training data only
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation ROC-AUC:", round(grid.best_score_, 3))

Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validation ROC-AUC: 0.767


In [23]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Logistic Regression
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Depth-3 Decision Tree
tree_cv_auc = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Random Forest
rf_cv_auc = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Tuned Random Forest
tuned_rf_cv_auc = grid.best_score_

cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "Cross-validation ROC-AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

print(cv_comparison.round(3))

                     Model  Cross-validation ROC-AUC
0      Logistic Regression                     0.774
1  Decision Tree (Depth 3)                     0.678
2            Random Forest                     0.747
3      Tuned Random Forest                     0.767


Logistic Regression was selected as the final model because it achieved the highest mean five-fold cross-validation ROC-AUC of 0.774 on the training data. The test set was not used for model selection.

In [24]:
# Generate out-of-fold predictions using 5-fold cross-validation
# Each training row gets a prediction from a model that did not train on that row

oof = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Display the number of generated probabilities
print("Number of predictions:", len(oof))


Number of predictions: 792


In [25]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Check two possible classification thresholds
for threshold in [0.50, 0.35]:

    # Flag patients whose predicted no-show probability

    flags = (oof >= threshold).astype(int)

    # Calculate precision:
    precision = precision_score(y_train, flags, zero_division=0)

    # Calculate recall:
    recall = recall_score(y_train, flags)

    # Calculate F1-score:
    f1 = f1_score(y_train, flags, zero_division=0)

    number_flagged = flags.sum()

    print("\nThreshold:", threshold)
    print("Precision:", round(precision, 3))
    print("Recall:", round(recall, 3))
    print("F1-score:", round(f1, 3))
    print("Number of patients flagged:", number_flagged)


Threshold: 0.5
Precision: 0.575
Recall: 0.249
F1-score: 0.347
Number of patients flagged: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.467
F1-score: 0.489
Number of patients flagged: 154


In [26]:
# Train the selected Logistic Regression pipeline
# The scaler is fitted using training data only
lr_pipe.fit(X_train, y_train)

# Predict no-show probabilities for the test data
probs = lr_pipe.predict_proba(X_test)[:, 1]

# Apply the selected threshold of 0.35
# Probability >= 0.35 means the appointment is flagged as high risk
test_flags = (probs >= 0.35).astype(int)

In [27]:
# Calculate precision on the test data
precision = precision_score(y_test, test_flags, zero_division=0)

# Calculate recall on the test data
recall = recall_score(y_test, test_flags)

# Calculate F1-score on the test data
f1 = f1_score(y_test, test_flags, zero_division=0)

# Calculate ROC-AUC using predicted probabilities
roc_auc = roc_auc_score(y_test, probs)

# Count the number of appointments flagged as high risk
number_flagged = test_flags.sum()

# Display final test results
print("Final Test Results")
print("-------------------")
print("Threshold:", 0.35)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("ROC-AUC:", round(roc_auc, 3))
print("Appointments flagged:", number_flagged)

Final Test Results
-------------------
Threshold: 0.35
Precision: 0.571
Recall: 0.571
F1-score: 0.571
ROC-AUC: 0.789
Appointments flagged: 49


In [28]:
# Import the csv module
import csv


with open("test_risk_scores.csv", "w", newline="") as file:

    # Create a CSV writer object
    writer = csv.writer(file)

    # Write the column headings
    writer.writerow([
        "appointment_id",
        "no_show",
        "risk_prob",
        "flag"
    ])

    # Loop through every test appointment
    for i in range(len(X_test)):

        # Get the original appointment ID using the test index
        appointment_id = df.loc[X_test.index[i], "appointment_id"]

        # Get the actual no-show status
        actual_status = y_test.iloc[i]

        # Get the predicted no-show probability
        risk_probability = probs[i]

        # Get the final risk flag (0 or 1)
        risk_flag = test_flags[i]

        # Write the appointment details as one CSV row
        writer.writerow([
            appointment_id,
            actual_status,
            risk_probability,
            risk_flag
        ])

# Confirm file creation
print("test_risk_scores.csv created successfully!")

test_risk_scores.csv created successfully!


In [29]:
# Open the saved CSV file in read mode
with open("test_risk_scores.csv", "r", newline="", encoding="utf-8") as file:

    reader = csv.reader(file)

    # Read all rows
    rows = list(reader)

    print("Total rows including header:", len(rows))


    print("Data records:", len(rows) - 1)

    print("Columns:", len(rows[0]))

    # Display first five data records
    print("First five records:")
    for row in rows[1:6]:
        print(row)

Total rows including header: 199
Data records: 198
Columns: 4
First five records:
['A0217', '0', '0.02156542041202344', '0']
['A0336', '0', '0.1509165920797874', '0']
['A0508', '0', '0.5186918193370788', '1']
['A0314', '0', '0.1913003349379444', '0']
['A0089', '0', '0.16771795103487128', '0']


# CareLoop Clinics — No-Show Guard: Final Conclusion

The project successfully developed a machine learning system to identify appointments at risk of being missed.

Data ingestion and cleaning produced 990 clean appointment records. SQL analysis identified differences in no-show rates across clinics and appointment lead-time groups.

Three machine learning approaches were evaluated: Logistic Regression, Decision Tree and Random Forest. Logistic Regression was selected based on its highest five-fold training cross-validation ROC-AUC of 0.774.

A classification threshold of 0.35 was selected using out-of-fold training predictions to improve the balance between precision and recall. The final model was evaluated on the held-out test set, and the resulting appointment risk scores were saved in `test_risk_scores.csv`.

The project has some limitations, including preprocessing leakage, possible patient overlap between training and test sets, and the use of a random rather than chronological split. These limitations should be addressed before deploying the model in a real clinic.

Overall, the CareLoop No-Show Guard demonstrates how data cleaning, SQL analytics and machine learning can help clinics identify potentially missed appointments and plan follow-up calls more effectively.